# Raw vs Translation-Canonical Discovery on Heat Data

This notebook compares two discovery paths on the same synthetic Heat fields:

- raw path: `to_pysindy_trajectories(...) -> fit_pysindy_discovery(...)`
- translation-canonical path: `build_translation_canonical_discovery_inputs(...) -> fit_pysindy_discovery(...)`

The goal is not to claim exact PDE recovery. The `v0.6` adapter returns backend-native PySINDy terms. The interesting question is whether heuristic translation canonicalization makes the fit structurally cleaner or more repeatable.


In [ ]:
import importlib.util

if importlib.util.find_spec("pysindy") is None:
    raise RuntimeError("Install pdelie[downstream] or pdelie[test] to run this notebook.")

import numpy as np

from pdelie import GeneratorFamily
from pdelie.data import generate_heat_1d_field_batch
from pdelie.discovery import (
    build_translation_canonical_discovery_inputs,
    fit_pysindy_discovery,
    to_pysindy_trajectories,
)


In [ ]:
def make_known_translation_generator() -> GeneratorFamily:
    basis_spec = {
        "variables": ["t", "x", "u"],
        "component_names": ["xi"],
        "basis_terms": [
            {"label": "1", "powers": [0, 0, 0]},
            {"label": "t", "powers": [1, 0, 0]},
            {"label": "x", "powers": [0, 1, 0]},
            {"label": "u", "powers": [0, 0, 1]},
        ],
        "component_ordering": ["xi"],
        "term_ordering": ["1", "t", "x", "u"],
        "layout": "component_major",
    }
    return GeneratorFamily(
        parameterization="polynomial_translation_affine",
        coefficients=np.array([[1.0, 0.0, 0.0, 0.0]], dtype=float),
        basis_spec=basis_spec,
        normalization="l2_unit",
        diagnostics={},
    )


def count_nonzero_backend_terms(result: dict[str, object], feature_name: str) -> int:
    return len(result["equation_terms"].get(feature_name, {}))


def run_raw_fit(field):
    trajectories, time_values, feature_names = to_pysindy_trajectories(field)
    result = fit_pysindy_discovery(trajectories, time_values, feature_names)
    return {
        "feature_names": feature_names,
        "trajectories": trajectories,
        "time_values": time_values,
        "fit": result,
    }


def run_canonical_fit(field, generator):
    inputs = build_translation_canonical_discovery_inputs(field, generator_family=generator)
    result = fit_pysindy_discovery(inputs["trajectories"], inputs["time_values"], inputs["feature_names"])
    return {
        "inputs": inputs,
        "fit": result,
    }


In [ ]:
generator = make_known_translation_generator()
field = generate_heat_1d_field_batch(batch_size=4, num_times=33, num_points=64, seed=100)

raw = run_raw_fit(field)
canonical = run_canonical_fit(field, generator)

primary_feature = raw["feature_names"][0]
summary = {
    "raw_status": raw["fit"]["status"],
    "canonical_status": canonical["fit"]["status"],
    "raw_coeff_shape": None if raw["fit"]["coefficients"] is None else tuple(raw["fit"]["coefficients"].shape),
    "canonical_coeff_shape": None if canonical["fit"]["coefficients"] is None else tuple(canonical["fit"]["coefficients"].shape),
    "raw_nonzero_primary_terms": count_nonzero_backend_terms(raw["fit"], primary_feature),
    "canonical_nonzero_primary_terms": count_nonzero_backend_terms(canonical["fit"], primary_feature),
    "canonical_alignment_policy": canonical["inputs"]["alignment_policy"],
    "canonical_alignment_shifts": canonical["inputs"]["alignment_shifts"][:4],
}
summary


## Repeatability across seeds

We compare the number of nonzero backend-native terms on the primary state feature across several Heat seeds.


In [ ]:
rows = []
for seed in [100, 101, 102, 103, 104]:
    field = generate_heat_1d_field_batch(batch_size=4, num_times=33, num_points=64, seed=seed)
    raw = run_raw_fit(field)
    canonical = run_canonical_fit(field, generator)
    feature_name = raw["feature_names"][0]
    rows.append(
        {
            "seed": seed,
            "raw_status": raw["fit"]["status"],
            "canonical_status": canonical["fit"]["status"],
            "raw_nonzero_primary_terms": count_nonzero_backend_terms(raw["fit"], feature_name),
            "canonical_nonzero_primary_terms": count_nonzero_backend_terms(canonical["fit"], feature_name),
        }
    )

rows


## Inspect one backend-native equation summary

These are backend-native PySINDy terms, not canonical PDE terms.


In [ ]:
print("primary feature:", primary_feature)
print("raw equation string:")
print(raw["fit"]["equation_strings"][primary_feature])
print()
print("canonical equation string:")
print(canonical["fit"]["equation_strings"][primary_feature])
